In [2]:
%load_ext autoreload
%autoreload 2

rename the files for the crater taking into consideration that there was a first run where not all trees where there

In [3]:
from man_seg_preproc.pretty_names import get_pretty_names, make_grid, rename_files

In [4]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

In [5]:
crowns_old_path = Path(r"C:/Users/smassaro/OneDrive - UGent/Documents/segmentation/the_crater/the_crater_crowns.gpkg")
crowns_new_path = Path(r"C:\Users\smassaro\OneDrive - UGent\Documents\segmentation\the_crater\the_crater\the_crater_crowns2.gpkg")
crowns_final_path = Path(r"C:\Users\smassaro\OneDrive - UGent\Documents\segmentation\the_crater\the_crater\the_crater_crowns_final.gpkg")

In [6]:
crowns_old = gpd.read_file(crowns_old_path)
crowns_old['segmented'] = crowns_old['segmented'].astype('boolean')
crowns_old.columns

Index(['GIS_ID', 'ID_number', 'area', 'area.1', 'neighbors',
       'neighbors_truncated', 'cell_id', 'grid_x', 'grid_y', 'grid_x_norm',
       'grid_y_norm', 'hilbert_dist', 'suffix', 'pretty_name', 'segmented',
       'notes', 'geometry'],
      dtype='object')

In [7]:
crowns_old.segmented

0       True
1       True
2      False
3       True
4       True
       ...  
339     True
340     True
341     True
342     True
343     True
Name: segmented, Length: 344, dtype: boolean

In [8]:
crowns_old.segmented.describe()

count      344
unique       2
top       True
freq       340
Name: segmented, dtype: object

In [9]:
crowns_new, cell_to_genus = get_pretty_names(crowns_new_path)

need to get the `segmented` and `notes` column from the old to the new

In [10]:
len(crowns_new), len(crowns_old)

(429, 344)

In [11]:
crowns_final = pd.merge(crowns_new, crowns_old[['pretty_name', 'segmented', 'notes']], how='left', on = "pretty_name")
crowns_final.columns, len(crowns_final)

(Index(['GIS_ID', 'ID_number', 'area', 'area.1', 'neighbors',
        'neighbors_truncated', 'geometry', 'cell_id', 'grid_x', 'grid_y',
        'grid_x_norm', 'grid_y_norm', 'hilbert_dist', 'suffix', 'pretty_name',
        'segmented', 'notes'],
       dtype='object'),
 429)

In [12]:
crowns_final.segmented.describe()

count      344
unique       2
top       True
freq       340
Name: segmented, dtype: object

In [13]:
crowns_final.segmented

0       <NA>
1       True
2       True
3      False
4       True
       ...  
424     True
425     True
426     True
427     True
428     <NA>
Name: segmented, Length: 429, dtype: boolean

In [14]:
crowns_final.segmented.dtype

BooleanDtype

In [15]:
crowns_final.to_file(crowns_final_path)

In [16]:
grid = make_grid(crowns_final, cell_to_genus)

In [17]:
grid.to_file(crowns_new_path.parent / "grid.gpkg")

for riscan pro convenience create a folder with all the non-segmented point clouds and the new names

In [18]:
start_folder = Path(r"C:\Users\smassaro\OneDrive - UGent\Documents\segmentation\the_crater\the_crater\selected_tree2")
target_folder = Path(r"C:\Users\smassaro\OneDrive - UGent\Documents\segmentation\the_crater\the_crater\selected_trees_diff")
target_folder.mkdir(exist_ok=True)

In [21]:
def rename_files(df, start_folder: Path, target_folder: Path, dry_run=False):
    name_map = dict(zip(df['GIS_ID'], df['pretty_name']))
    las_files = list(start_folder.glob('*.las'))
    for f in las_files:
        stem = f.stem
        if stem in name_map:
            new_name = target_folder / f"{name_map[stem]}.las"
            print(f"{f.name} -> {new_name.name}")
            if not dry_run: f.rename(new_name)
        else: print(f"Skipping {f.name} (not in map)")

In [23]:
df = crowns_final[crowns_final.segmented.isna()]

In [ ]:
# rename_files(crowns_final[crowns_final.segmented.isna()], start_folder, target_folder, dry_run=False)

CRA01-01_338470_8073610_10.las -> Aata_10.las
Skipping CRA01-01_338470_8073610_20_1.las (not in map)
Skipping CRA01-01_338470_8073620_01_1.las (not in map)
Skipping CRA01-01_338470_8073620_04.las (not in map)
Skipping CRA01-01_338470_8073620_05.las (not in map)
Skipping CRA01-01_338470_8073620_07_1.las (not in map)
Skipping CRA01-01_338470_8073620_08.las (not in map)
Skipping CRA01-01_338470_8073620_09.las (not in map)
CRA01-01_338470_8073620_10_1.las -> Abax_10_1.las
Skipping CRA01-01_338470_8073620_17_1.las (not in map)
Skipping CRA01-01_338470_8073620_18_b.las (not in map)
CRA01-01_338470_8073630_05_1.las -> Lama_05_1.las
Skipping CRA01-01_338470_8073630_07_1.las (not in map)
Skipping CRA01-01_338470_8073630_10.las (not in map)
Skipping CRA01-01_338470_8073630_12.las (not in map)
Skipping CRA01-01_338470_8073630_19_1.las (not in map)
Skipping CRA01-01_338470_8073630_22.las (not in map)
Skipping CRA01-01_338470_8073630_26_d.las (not in map)
Skipping CRA01-01_338470_8073630_28.las (no